# 00 — Setup do ambiente no Google Colab

Execute no início de cada sessão do Colab (kernel remoto via extensão do VS Code, v0.2.1+).

O kernel remoto **não enxerga os arquivos locais**. Há dois modos para levar o código até ele:

- **`upload`** (desenvolvimento): no Explorer do VS Code, clique com o botão direito em `src/`, `configs/` e `pyproject.toml` → *Upload to Colab* (destino: `/content/ProjetoFinalRNAP_UnB2026/`). Reenvie após cada alteração e confira com `!ls`. Não há hash de commit rastreável.
- **`git`** (experimentos que entram no artigo): faça *commit + push* localmente; o notebook clona/atualiza o repositório público. O hash do commit é registrado nos resultados.


In [ ]:
MODE = "upload"  # "upload" ou "git"
REPO_URL = "https://github.com/Diehlgit/ProjetoFinalRNAP_UnB2026.git"  # repositório público, sem token
BRANCH = "main"
REPO_DIR = "/content/ProjetoFinalRNAP_UnB2026"

## 1. Montar o Google Drive

Também disponível pela paleta de comandos: `Colab: Mount Google Drive to Server...`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Disponibilizar o código

In [ ]:
import os, subprocess

if MODE == "git":
    if not os.path.isdir(f"{REPO_DIR}/.git"):
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
    print(subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"],
                         capture_output=True, text=True).stdout)
elif MODE == "upload":
    missing = [p for p in ("src/cerrado_flora", "configs", "pyproject.toml")
               if not os.path.exists(f"{REPO_DIR}/{p}")]
    assert not missing, f"Faltam arquivos em {REPO_DIR}: {missing}. Use 'Upload to Colab'."
    print("Modo upload: resultados ficarão sem hash de commit (git_commit = None).")
else:
    raise ValueError(MODE)

## 3. Instalar o pacote (modo editável)

`torch`/`torchvision` já vêm no Colab e não são reinstalados. No modo `upload`, reexecute após reenviar `src/` apenas se mudar dependências; o modo editável já reflete as alterações (reinicie o kernel ou use `importlib.reload`).

In [ ]:
%pip install -q -e {REPO_DIR}

## 4. Verificar ambiente e caminhos

In [ ]:
import yaml
from pathlib import Path
from cerrado_flora.utils import run_metadata

with open(f"{REPO_DIR}/configs/paths.yaml") as f:
    paths = yaml.safe_load(f)
drive_root = Path(paths["drive_root"])
for name, sub in paths["drive_subdirs"].items():
    p = drive_root / sub
    p.mkdir(parents=True, exist_ok=True)
    print(f"{name:12s} {p}")

meta = run_metadata.collect(REPO_DIR)
print({k: v for k, v in meta.items() if k != "pip_freeze"})